# Comparison of aggregation

This notebook compares the 3 main methods of aggregation (Rank, Mean and Max).

Results are explained in Section 4.4.

In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
from utils import load_nested_results

all_results = load_nested_results("results/")

In [3]:
from itertools import combinations
from sklearn.metrics import average_precision_score
import numpy as np
from pyod.utils.utility import standardizer
from scipy.stats import rankdata

scores_all_types = {}

for dataset_name, results in all_results.items():

    score_dataset = {}
    model_names = list(results.keys() - {"ground_truth"})
    n_fold = len(results["ground_truth"])

    for model in model_names:
        scores_model = []
        for fold in range(n_fold):
            scores_model.append(
                average_precision_score(
                    results["ground_truth"][fold], results[model][fold]["scores"]
                )
            )
        score_dataset[model] = scores_model

    for comb in combinations(range(len(model_names)), 3):

        name1 = model_names[comb[0]]
        name2 = model_names[comb[1]]
        name3 = model_names[comb[2]]

        scores_model_rank = []
        scores_model_max = []
        scores_model_mean = []
        scores_model_median = []
        scores_model_min = []

        for fold in range(n_fold):
            y_true = results["ground_truth"][fold]

            scores_1 = results[name1][fold]["scores"]
            scores_2 = results[name2][fold]["scores"]
            scores_3 = results[name3][fold]["scores"]

            # Rank-based fusion
            ranks = [
                rankdata(score, "average") for score in [scores_1, scores_2, scores_3]
            ]
            auc_rank = average_precision_score(y_true, np.mean(ranks, axis=0))
            scores_model_rank.append(auc_rank)

            # Standardize once to avoid redundant computations
            scores_triplet_std = standardizer(np.stack([scores_1, scores_2, scores_3]))

            # Max fusion
            scores_agreges_max = np.max(scores_triplet_std, axis=0)
            scores_model_max.append(average_precision_score(y_true, scores_agreges_max))

            # Mean fusion
            scores_agreges_mean = np.mean(scores_triplet_std, axis=0)
            scores_model_mean.append(
                average_precision_score(y_true, scores_agreges_mean)
            )

            # Median fusion
            scores_agreges_median = np.median(scores_triplet_std, axis=0)
            scores_model_median.append(
                average_precision_score(y_true, scores_agreges_median)
            )

            # Minimum fusion
            scores_agreges_min = np.min(scores_triplet_std, axis=0)
            scores_model_min.append(average_precision_score(y_true, scores_agreges_min))

        score_dataset[f"{name1}-{name2}-{name3}_rank"] = scores_model_rank
        score_dataset[f"{name1}-{name2}-{name3}_max"] = scores_model_max
        score_dataset[f"{name1}-{name2}-{name3}_mean"] = scores_model_mean
        score_dataset[f"{name1}-{name2}-{name3}_median"] = scores_model_median
        score_dataset[f"{name1}-{name2}-{name3}_min"] = scores_model_min

    scores_all_types[dataset_name] = score_dataset

In [4]:
n_emsemble_rank = 0
n_emsemble_max = 0
n_emsemble_mean = 0

for model_name in scores_all_types[dataset_name].keys():
    if "_rank" in model_name:
        n_emsemble_rank += 1
    elif "_max" in model_name:
        n_emsemble_max += 1
    elif "_mean" in model_name:
        n_emsemble_mean += 1
    elif "_median" in model_name:
        n_emsemble_mean += 1
    elif "_min" in model_name:
        n_emsemble_mean += 1

In [7]:
import pandas as pd

all_res = {}


for dataset_name, dataset_scores in scores_all_types.items():
    means = np.zeros(6)

    for model_name, scores in dataset_scores.items():
        mean_score = np.max(scores)
        if "_rank" in model_name:
            means[1] += mean_score
        elif "_max" in model_name:
            means[2] += mean_score
        elif "_mean" in model_name:
            means[3] += mean_score
        elif "_median" in model_name:
            means[4] += mean_score
        elif "_min" in model_name:
            means[5] += mean_score
        else:
            means[0] += mean_score

    means[1] /= 364
    means[2] /= 364
    means[3] /= 364
    means[4] /= 364
    means[5] /= 364
    means[0] /= len(model_names)

    all_res[dataset_name] = means


df = (
    pd.DataFrame(
        all_res,
        index=["Unique", "Rank", "Max", "Mean", "Median", "Min"],
    )
    .round(2)
    .T
)

df[["Rank", "Max", "Mean", "Median", "Min"]] *= 100
df[["Rank", "Max", "Mean", "Median", "Min"]].round()

,Rank,Max,Mean,Median,Min
2_annthyroid,26.0,21.0,9.0,13.0,21.0
4_breastw,98.0,59.0,48.0,52.0,59.0
14_glass,18.0,22.0,19.0,26.0,22.0
15_Hepatitis,84.0,64.0,47.0,41.0,64.0
21_Lymphography,100.0,68.0,37.0,38.0,68.0
23_mammography,30.0,8.0,4.0,6.0,8.0
27_PageBlocks,60.0,35.0,12.0,21.0,35.0
29_Pima,55.0,45.0,42.0,43.0,45.0
37_Stamps,62.0,35.0,22.0,23.0,35.0
38_thyroid,58.0,28.0,5.0,11.0,28.0


In [8]:
# print(df.mean().to_latex(float_format="%.0f"))
df[["Rank", "Max", "Mean", "Median", "Min"]].mean().round()

Rank      52.0
Max       35.0
Mean      24.0
Median    27.0
Min       36.0
dtype: float64